In [1]:
#import current working directory

import os
import sys

print(os.getcwd())

c:\Users\Greesha Vaishnavi\Desktop\dsprojects\E_Commerce_Customer_Segmentation\research


In [2]:
# Read the sys path
sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\E_Commerce_Customer_Segmentation\src")
sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\E_Commerce_Customer_Segmentation")

In [3]:
# present working directory

%pwd

'c:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\E_Commerce_Customer_Segmentation\\research'

In [4]:
# move to parent directory

os.chdir("../") 

In [5]:
# present working directory

%pwd

'c:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\E_Commerce_Customer_Segmentation'

In [6]:
# test project import

import box

print(box.__version__)

7.4.1


In [7]:
# entity 

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_file: Path
    transformed_data_file: Path

In [8]:
from E_Commerce_Customer_Segmentation.constant import *
from E_Commerce_Customer_Segmentation.utils.common import read_yaml,create_directories
from E_Commerce_Customer_Segmentation.entity.config_entity import DataTransformationConfig

In [9]:
# configuration manager


class ConfigurationManager:

    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    def get_data_transformation_config(self) -> DataTransformationConfig:

        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=Path(config.root_dir),
            data_file=Path(config.data_file),
            transformed_data_file=Path(config.transformed_data_file)
        )

        return data_transformation_config

In [10]:
import os
import pandas as pd

from E_Commerce_Customer_Segmentation.logging import logger
from E_Commerce_Customer_Segmentation.entity.config_entity import DataTransformationConfig


In [11]:
# components


class DataTransformation:

    def __init__(self, config: DataTransformationConfig):

        self.config = config

    def transform_data(self):

        logger.info("Starting Data Transformation")

        # Read dataset

        df = pd.read_excel(self.config.data_file)

        logger.info(f"Original dataset shape: {df.shape}")

        # Remove duplicate rows

        df = df.drop_duplicates()

        logger.info(f"Shape after removing duplicates: {df.shape}")

        # Remove transactions without CustomerID

        df = df.dropna(subset=["CustomerID"])

        logger.info(f"Shape after removing missing CustomerID: {df.shape}")

        # Remove cancelled invoices

        df = df[
            ~df["InvoiceNo"]
            .astype(str)
            .str.startswith("C")
        ]

        logger.info(f"Shape after removing cancelled invoices: {df.shape}")

        # Keep positive quantities

        df = df[df["Quantity"] > 0]

        logger.info(f"Shape after removing invalid quantities: {df.shape}")

        # Keep positive unit prices

        df = df[df["UnitPrice"] > 0]

        logger.info(f"Shape after removing invalid prices: {df.shape}")

        # Create Revenue

        df["Revenue"] = (
            df["Quantity"] *
            df["UnitPrice"]
        )

        # Reference date for Recency

        reference_date = (
            df["InvoiceDate"].max()
            + pd.Timedelta(days=1)
        )

        # Create RFM dataset

        rfm = df.groupby("CustomerID").agg(
            Recency=(
                "InvoiceDate",
                lambda x:
                (reference_date - x.max()).days
            ),

            Frequency=(
                "InvoiceNo",
                "nunique"
            ),

            Monetary=(
                "Revenue",
                "sum"
            )
        )

        # Reset index

        rfm = rfm.reset_index()

        logger.info(f"Customer RFM shape: {rfm.shape}")

        # Save transformed data

        os.makedirs(self.config.root_dir,
            exist_ok=True
        )

        rfm.to_csv(
            self.config.transformed_data_file,
            index=False
        )

        logger.info(
            f"Transformed data saved to: "
            f"{self.config.transformed_data_file}"
        )

        return rfm

In [12]:
# pipeline 


class DataTransformationTrainingPipeline:

    def __init__(self):
        pass

    def main(self):

        try:

            logger.info(">>>>>> Data Transformation Stage Started <<<<<<")

            config = ConfigurationManager()

            data_transformation_config = (config.get_data_transformation_config())

            data_transformation = DataTransformation(config=data_transformation_config)

            data_transformation.transform_data()

            logger.info(">>>>>> Data Transformation Stage Completed Successfully <<<<<<")

        except Exception as e:

            logger.exception(e)

            raise e

In [13]:
obj = DataTransformationTrainingPipeline()
obj.main()

[2026-08-31 07:35:22,915: INFO: 2337213410: >>>>>> Data Transformation Stage Started <<<<<<]
[2026-08-31 07:35:22,921: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-08-31 07:35:22,925: INFO: common: yaml file: params.yaml loaded successfully]
[2026-08-31 07:35:22,932: INFO: common: created directory at: artifacts]
[2026-08-31 07:35:22,937: INFO: common: created directory at: artifacts/data_transformation]
[2026-08-31 07:35:22,939: INFO: 3993600415: Starting Data Transformation]
[2026-08-31 07:36:09,369: INFO: 3993600415: Original dataset shape: (541909, 8)]
[2026-08-31 07:36:09,718: INFO: 3993600415: Shape after removing duplicates: (536641, 8)]
[2026-08-31 07:36:09,819: INFO: 3993600415: Shape after removing missing CustomerID: (401604, 8)]
[2026-08-31 07:36:10,052: INFO: 3993600415: Shape after removing cancelled invoices: (392732, 8)]
[2026-08-31 07:36:10,074: INFO: 3993600415: Shape after removing invalid quantities: (392732, 8)]
[2026-08-31 07:36:10,106: I